In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import holidays
import tools

In [2]:

df_fi_hard = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard.parquet")

## Jak wykluczyc produkty( TowId) które nie są sprzedawane od jakiegoś czasu

potrzeba policzyć datę ostatniej sprzedaży per TowId i porównać ją z datą odniesienia (koniec danych albo dzisiejsza data). Produkty, które "milczą" dłużej niż przyjęty próg  oznaczasz jako dead stock / do wykluczenia.


In [3]:
# ============================================================
# Ostatnia sprzedaż per TowId + wykluczenie "martwych" produktów
# ============================================================

# 1. Tylko rekordy sprzedaży
df_sprzedaz = df_fi_hard[df_fi_hard['TypRuchu'] == 'sprzedaz'].copy()

# 2. Data odniesienia — koniec zakresu danych (albo pd.Timestamp.now(), jeśli wolisz "dziś")
data_odniesienia = df_fi_hard['Data'].max()
print(f"Data odniesienia: {data_odniesienia}")

# 3. Ostatnia sprzedaż per TowId
ostatnia_sprzedaz = (
    df_sprzedaz
    .groupby('TowId')['Data']
    .max()
    .reset_index()
    .rename(columns={'Data': 'OstatniaSprzedaz'})
)

ostatnia_sprzedaz['DniOdOstatniejSprzedazy'] = (
    data_odniesienia - ostatnia_sprzedaz['OstatniaSprzedaz']
).dt.days


Data odniesienia: 2026-01-31 00:00:00


In [4]:
#  Progi rotacji wyliczone z percentyli rzeczywistego rozkładu

percentyle = ostatnia_sprzedaz['DniOdOstatniejSprzedazy'].quantile(
    [0.10, 0.25, 0.50, 0.75, 0.80, 0.90, 0.95, 0.99]
)
print(percentyle)

0.10       0.0
0.25       3.0
0.50     101.0
0.75     543.0
0.80     655.0
0.90     879.0
0.95    1013.0
0.99    1102.0
Name: DniOdOstatniejSprzedazy, dtype: float64


In [5]:
# 4. Próg "martwego" produktu obliczone poniżej
prog_dni = 550 # patrz komórka wyżej

towid_martwe = ostatnia_sprzedaz[
    ostatnia_sprzedaz['DniOdOstatniejSprzedazy'] > prog_dni
]['TowId']

print(f"TowId z ostatnią sprzedażą >{prog_dni} dni temu: {len(towid_martwe)} pozycji")

# 5. TowId, które W OGÓLE nie mają sprzedaży w danych (np. tylko przyjęcia, zwroty itp.)
wszystkie_towid = df_fi_hard['TowId'].unique()
towid_bez_sprzedazy = set(wszystkie_towid) - set(ostatnia_sprzedaz['TowId'])
print(f"TowId bez żadnej sprzedaży w okresie: {len(towid_bez_sprzedazy)} pozycji")

# 6. Finalna lista do wykluczenia
towid_do_wykluczenia = set(towid_martwe) | towid_bez_sprzedazy

# 6. Finalna lista do wykluczenia - tylko TowId bez żadnej sprzedaży
df_fi_soft = df_fi_hard[~df_fi_hard['TowId'].isin(towid_bez_sprzedazy)].copy()

print(f"Przed: {len(df_fi_hard):,} rekordów, {df_fi_hard['TowId'].nunique():,} TowId pozycji")
print(f"Po:    {len(df_fi_soft):,} rekordów, {df_fi_soft['TowId'].nunique():,} TowId pozycji")
print(f"Wykluczono: {len(towid_bez_sprzedazy):,} TowId pozycji")
df_fi_soft.to_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet", index=False)


TowId z ostatnią sprzedażą >550 dni temu: 3097 pozycji
TowId bez żadnej sprzedaży w okresie: 7211 pozycji
Przed: 4,106,901 rekordów, 19,692 TowId pozycji
Po:    4,057,332 rekordów, 12,481 TowId pozycji
Wykluczono: 7,211 TowId pozycji


In [6]:
# Suma kontrolna (hash) danych w pliku Parquet, z uwzględnieniem sortowania po kolumnach TowId i Data.

nazwa_pliku = "fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet   8dd8ce7da71466f6ef5f128d5449b87e4bc4675ea71f4a08f6e919e48e61765e
